In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
!pip install evaluate rouge-score nltk

In [ ]:
!pip install bert_score

In [29]:
# Imports 
from transformers import Gemma2ForCausalLM, AutoTokenizer,  BitsAndBytesConfig
from peft import PeftModel
import shutil
import os 
import torch
from kaggle_secrets import UserSecretsClient
import evaluate
from datasets import Dataset
import pandas as pd
import time 
import numpy as np

In [2]:
shutil.copytree(
    "/kaggle/input/datasets/rpatel29/gemma-args-2/checkpoint-2400",
    "/kaggle/working/checkpoint-2400"
)

'/kaggle/working/checkpoint-2400'

In [3]:
shutil.copytree(
    "/kaggle/input/datasets/rpatel29/gemma-args-1/checkpoint-800",
    "/kaggle/working/checkpoint-800"
)

'/kaggle/working/checkpoint-800'

In [4]:
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret('HF_TOKEN')

config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Load the model's tokenizer and the pre-trained model
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-9b-it", token=secret_value_0)
base_model = Gemma2ForCausalLM.from_pretrained("google/gemma-2-9b-it", 
                                               quantization_config=config, 
                                               device_map="auto",
                                               token=secret_value_0
                                              )

config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

In [5]:
# Load in LoRA weights from final model
model = PeftModel.from_pretrained(
    base_model,
    "/kaggle/working/checkpoint-2400"    
)
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma2ForCausalLM(
      (model): Gemma2Model(
        (embed_tokens): Embedding(256000, 3584, padding_idx=0)
        (layers): ModuleList(
          (0-41): 42 x Gemma2DecoderLayer(
            (self_attn): Gemma2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
           

In [6]:
def generate_checkpoint_2400(prompt):

    message = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    model_input = tokenizer.apply_chat_template(
        message,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    input_device = model.get_input_embeddings().weight.device

    model_input = {
        key: value.to(input_device)
        for key, value in model_input.items()
    }

    input_length = model_input["input_ids"].shape[-1]
    
    outputs = model.generate(
        **model_input,
        max_new_tokens=318,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        num_beams=4,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3
    )

    generated_tokens = outputs[0][input_length:]
    
    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )
    

In [7]:
prompt = (
    "Task: medical_qa\n"
    "Instruction: Answer the patient's medical question using the provided medical context. "
    "Do not diagnose the patient and do not assume that symptoms alone confirm a condition.\n"
    "Patient: I've been feeling tired and nauseous lately. Could this mean I have HIV?\n"
    "Medical context: Symptoms such as fatigue and nausea are not specific to HIV and cannot determine whether someone has HIV. "
    "HIV testing is needed to determine whether infection is present. The timing and type of test can affect when HIV infection can be detected.\n"
)
generate(prompt)

"Fatigue and nausea alone don't confirm HIV. To know for sure, you'd need an HIV test. The type and timing of the test matter because they influence when you can get an accurate result. If you're concerned about HIV, it's important to talk to a healthcare professional about getting tested. They can help you understand the different types of tests available and which one might be right for you based on when you last had potential exposure to HIV. You can also find more information about HIV testing on the Centers for Disease Control and Prevention (CDC) website at https://www.cdc.gov/hiv/testing/.\n"

In [8]:
prompt = (
    "Task: patient_chat\n"
    "Instruction: Respond like a calm and empathetic medical assistant. Acknowledge user's feelings, be supportive, and explain things clearly in a natural, conversational way.\n"
    "Context:\n"
    "Chest pressure that occurs during physical exertion and improves with rest can be a sign of angina. Chest pain or pressure should be medically evaluated, especially when it occurs with exertion.\n"
    "Patient: I sometimes get a tight pressure in my chest when I walk upstairs. It goes away when I rest. Should I be worried?\n"
    "Response:"
)
generate(prompt)

'That’s completely understandable to wonder about. Chest pressure that happens during exercise and gets better when you rest could be angina. It’s important to check with a healthcare professional, especially if you’re experiencing chest pain or discomfort.\n'

In [9]:
prompt = (
    "Task: patient_chat\n"
    "Instruction: Respond like a calm and empathetic medical assistant. "
    "Acknowledge user's feelings, be supportive, and explain things clearly "
    "in a natural, conversational way.\n"
    "Context:\n"
    "Leftover antibiotics from a previous illness should not be used for a new "
    "illness unless a healthcare professional specifically recommends them. "
    "Antibiotics do not treat viral infections, and many sore throats are caused "
    "by viruses. The appropriate treatment depends on the cause of the sore throat "
    "and the specific medication involved. A healthcare professional can determine "
    "whether testing or treatment is needed.\n"
    "Patient: I have some leftover antibiotics from last year. I have a sore throat right now. Should I just start taking them?\n"
    "Response:"
)
generate(prompt)

'It’s completely understandable that you’d wonder about this. Leftover antibiotics shouldn’t be used unless your healthcare provider says it’s okay. It’s important to follow their guidance on how to manage your sore throat.\n'

In [10]:
prompt = (
    "Task: patient_chat\n"
    "Instruction: Respond like a calm and empathetic medical assistant. Acknowledge user's feelings, be supportive, and explain things clearly in a natural, conversational way.\n"
    "Context:\n"
    "Headaches are common and can have many causes, and most headaches are not caused by a serious illness. A new or persistent headache pattern should be evaluated by a healthcare professional. Headaches associated with brain tumors can sometimes be worse in the morning, but a headache by itself does not mean that someone has a brain tumor. Other concerning symptoms can include worsening headaches, seizures, vision changes, weakness or numbness, balance problems, confusion, or repeated vomiting.\n"
    "Patient: I've been having headaches every morning for the past 2 weeks. I Googled it and I'm really scared it might be a brain tumor. Is that likely?\n"
    "Response:"
)
generate(prompt)

'You’re not alone in thinking about this. Many people experience headaches, and they can have various causes. Most headaches aren’t serious. If you’ve noticed a change in your headache patterns, it’s a good idea to check in with a healthcare provider. They can help figure out what’s going on.\n'

**Evaluation of Checkpoint-2400**

In [13]:
def preprocess(data):
    
    user_message = (
        f"Task: {data['task_type']}\n"
        f"Instruction: {data['instruction']}\n"
        f"Patient: {data['input']}"
    )

    messages = [
        {
            "role": "user",
            "content": user_message
        }
    ]

    model_input = tokenizer.apply_chat_template(
        messages,
        tokenize=True, 
        add_generation_prompt=True,
        return_tensors="pt"
    )

    return {
        "input_ids": model_input["input_ids"][0].tolist(),
        "attention_mask": model_input["attention_mask"][0].tolist()
    }

    

In [14]:
# Load in the test set 
test_set = pd.read_csv('/kaggle/input/datasets/rpatel29/chatbot-data/completed-data-2/test/test.csv')
test_data = Dataset.from_pandas(test_set)
test_dataset = test_data.map(preprocess)

Map:   0%|          | 0/5325 [00:00<?, ? examples/s]

In [15]:
rouge = evaluate.load('rouge')
predictions = []
references = []

sample = test_dataset.shuffle(seed=42).select(range(20))

start = time.time()

# Fill up the references and candidate sets 
for i, example in enumerate(sample, start=1):
    references.append(example["output"])

    user_message = (
        f"Task: {example['task_type']}\n"
        f"Instruction: {example['instruction']}\n"
        f"Patient: {example['input']}"
    )

    response = generate_checkpoint_2400(user_message)

    predictions.append(response)

    elapsed = time.time() - start
    avg_time = elapsed / i

    print(
        f"{i}/20 | "
        f"Avg: {avg_time:.2f}s/example | "
        f"Estimated total: {avg_time * len(test_dataset) / 3600:.2f}h"
    )

1/20 | Avg: 33.23s/example | Estimated total: 49.15h
2/20 | Avg: 40.44s/example | Estimated total: 59.81h
3/20 | Avg: 32.24s/example | Estimated total: 47.69h
4/20 | Avg: 35.50s/example | Estimated total: 52.50h
5/20 | Avg: 30.94s/example | Estimated total: 45.76h
6/20 | Avg: 30.11s/example | Estimated total: 44.54h
7/20 | Avg: 32.02s/example | Estimated total: 47.37h
8/20 | Avg: 34.85s/example | Estimated total: 51.55h
9/20 | Avg: 32.84s/example | Estimated total: 48.57h
10/20 | Avg: 36.04s/example | Estimated total: 53.31h
11/20 | Avg: 35.55s/example | Estimated total: 52.59h
12/20 | Avg: 35.96s/example | Estimated total: 53.19h
13/20 | Avg: 35.73s/example | Estimated total: 52.85h
14/20 | Avg: 34.77s/example | Estimated total: 51.43h
15/20 | Avg: 33.94s/example | Estimated total: 50.20h
16/20 | Avg: 33.36s/example | Estimated total: 49.35h
17/20 | Avg: 32.64s/example | Estimated total: 48.29h
18/20 | Avg: 31.85s/example | Estimated total: 47.11h
19/20 | Avg: 31.92s/example | Estimat

In [21]:
results = rouge.compute(predictions=predictions, references=references)
print("CHECKPOINT-2400 ROUGE SCORE:\n")

print(pd.DataFrame([results]))

CHECKPOINT-2400 ROUGE SCORE:

     rouge1    rouge2   rougeL  rougeLsum
0  0.321101  0.130986  0.23263   0.232556


In [30]:
bert = evaluate.load("bertscore")
result = bert.compute(predictions=predictions, references=references, lang="en")
print("CHECKPOINT-2400 BERT SCORE:\n")

precision = np.array(result['precision'])
recall = np.array(result['recall'])
f1 = np.array(result['f1'])

print(f"System Level Precision: {precision.mean().item():.4f}")
print(f"System Level Recall: {recall.mean().item():.4f}")
print(f"System Level F1-Score: {f1.mean().item():.4f}")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


CHECKPOINT-2400 BERT SCORE:

System Level Precision: 0.8757
System Level Recall: 0.8643
System Level F1-Score: 0.8698


**Evaluation of checkpoint-800**

In [31]:
# Load in LoRA weights from final model
model2 = PeftModel.from_pretrained(
    base_model,
    "/kaggle/working/checkpoint-800"    
)
model2.eval()

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma2ForCausalLM(
      (model): Gemma2Model(
        (embed_tokens): Embedding(256000, 3584, padding_idx=0)
        (layers): ModuleList(
          (0-41): 42 x Gemma2DecoderLayer(
            (self_attn): Gemma2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
           

In [32]:
def generate_checkpoint_800(prompt):
    message = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    model_input = tokenizer.apply_chat_template(
        message,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    input_device = model2.get_input_embeddings().weight.device

    model_input = {
        key: value.to(input_device)
        for key, value in model_input.items()
    }

    input_length = model_input["input_ids"].shape[-1]
    
    outputs = model2.generate(
        **model_input,
        max_new_tokens=318,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        num_beams=4,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3
    )

    generated_tokens = outputs[0][input_length:]
    
    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

In [33]:
predictions = []
references = []

sample = test_dataset.shuffle(seed=42).select(range(20))

start = time.time()

# Fill up the references and candidate sets 
for i, example in enumerate(sample, start=1):
    references.append(example["output"])

    user_message = (
        f"Task: {example['task_type']}\n"
        f"Instruction: {example['instruction']}\n"
        f"Patient: {example['input']}"
    )

    response = generate_checkpoint_800(user_message)

    predictions.append(response)

    elapsed = time.time() - start
    avg_time = elapsed / i

    print(
        f"{i}/20 | "
        f"Avg: {avg_time:.2f}s/example | "
        f"Estimated total: {avg_time * len(test_dataset) / 3600:.2f}h"
    )

1/20 | Avg: 28.53s/example | Estimated total: 42.20h
2/20 | Avg: 23.96s/example | Estimated total: 35.45h
3/20 | Avg: 20.74s/example | Estimated total: 30.68h
4/20 | Avg: 25.35s/example | Estimated total: 37.49h
5/20 | Avg: 24.00s/example | Estimated total: 35.49h
6/20 | Avg: 26.68s/example | Estimated total: 39.46h
7/20 | Avg: 28.48s/example | Estimated total: 42.13h
8/20 | Avg: 29.83s/example | Estimated total: 44.13h
9/20 | Avg: 28.23s/example | Estimated total: 41.76h
10/20 | Avg: 27.92s/example | Estimated total: 41.29h
11/20 | Avg: 27.09s/example | Estimated total: 40.07h
12/20 | Avg: 27.67s/example | Estimated total: 40.93h
13/20 | Avg: 26.57s/example | Estimated total: 39.30h
14/20 | Avg: 26.07s/example | Estimated total: 38.56h
15/20 | Avg: 25.77s/example | Estimated total: 38.12h
16/20 | Avg: 25.63s/example | Estimated total: 37.91h
17/20 | Avg: 25.05s/example | Estimated total: 37.05h
18/20 | Avg: 24.75s/example | Estimated total: 36.60h
19/20 | Avg: 24.74s/example | Estimat

In [34]:
results = rouge.compute(predictions=predictions, references=references)
print("CHECKPOINT-800 ROUGE SCORE:\n")

print(pd.DataFrame([results]))

CHECKPOINT-800 ROUGE SCORE:

     rouge1    rouge2    rougeL  rougeLsum
0  0.276772  0.083851  0.184189   0.183408


In [35]:
result = bert.compute(predictions=predictions, references=references, lang="en")
print("CHECKPOINT-800 BERT SCORE:\n")

precision = np.array(result['precision'])
recall = np.array(result['recall'])
f1 = np.array(result['f1'])

print(f"System Level Precision: {precision.mean().item():.4f}")
print(f"System Level Recall: {recall.mean().item():.4f}")
print(f"System Level F1-Score: {f1.mean().item():.4f}")

CHECKPOINT-800 BERT SCORE:

System Level Precision: 0.8810
System Level Recall: 0.8545
System Level F1-Score: 0.8673
